In [1]:
from dataclasses import dataclass
from pathlib import Path
import json
import sys
import time

import pandas as pd

PROJECT_ROOT = Path(r"E:\slde-aft")
SRC_PATH = PROJECT_ROOT / "src"
OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "dec003_probkb_local_test"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from pkb_math import (
    DEFAULT_SHRINKAGE,
    conservative_noisy_or,
    final_confidence,
)

print("Standalone DEC-003 test setup complete.")
print("Output directory:", OUTPUT_DIR)
print("Shrinkage:", DEFAULT_SHRINKAGE)

Standalone DEC-003 test setup complete.
Output directory: E:\slde-aft\outputs\dec003_probkb_local_test
Shrinkage: 0.75


In [2]:
@dataclass
class LocalTriple:
    subject: str
    predicate: str
    object: str
    confidence: float
    source_id: str
    source_type: str
    provenance: str
    extractor_version: str = "local_test"

    def key(self):
        return (
            self.subject.strip().lower(),
            self.predicate.strip().lower(),
            self.object.strip().lower(),
        )


class LocalProbabilisticKB:
    def __init__(self):
        self.accepted = {}
        self.candidates = {}
        self.history = []

print("Local test classes defined.")

Local test classes defined.


In [3]:
policy_path = (
    PROJECT_ROOT
    / "configs"
    / "functional_predicates_dec003_controlled.json"
)

with policy_path.open("r", encoding="utf-8") as file:
    policy = json.load(file)

FUNCTIONAL_PREDICATES = {
    value.strip().lower()
    for value in policy["functional_predicates"]
}

print("Functional predicates:")
print(sorted(FUNCTIONAL_PREDICATES))

Functional predicates:
['belongs_to_category', 'has_color', 'has_noise_cancellation', 'has_price_usd', 'target_market_region']


In [4]:
test_triples = [
    LocalTriple(
        subject="Product-A",
        predicate="has_noise_cancellation",
        object="yes",
        confidence=0.90,
        source_id="local-source-1",
        source_type="unstructured",
        provenance="Product-A supports noise cancellation.",
    ),
    LocalTriple(
        subject="Product-A",
        predicate="has_noise_cancellation",
        object="yes",
        confidence=0.92,
        source_id="local-source-2",
        source_type="unstructured",
        provenance="Noise cancellation is included for Product-A.",
    ),
    LocalTriple(
        subject="Product-A",
        predicate="has_noise_cancellation",
        object="no",
        confidence=0.70,
        source_id="local-source-3",
        source_type="unstructured",
        provenance="Conflicting local test observation.",
    ),
    LocalTriple(
        subject="Product-B",
        predicate="made_of_material",
        object="aluminum",
        confidence=0.85,
        source_id="local-source-4",
        source_type="structured",
        provenance="Structured material field: aluminum.",
    ),
]

print("Manual observations:", len(test_triples))
print("API calls: 0")

Manual observations: 4
API calls: 0


In [5]:
def is_functional(predicate):
    return predicate.strip().lower() in FUNCTIONAL_PREDICATES


def collect_confidences(records, key):
    return [
        float(record["confidence"])
        for record in records
        if record["triple_key"] == key
    ]


def calculate_local_scores(records):
    rows = []

    slot_objects = {}

    for record in records:
        slot_key = (
            record["subject"].strip().lower(),
            record["predicate"].strip().lower(),
        )

        slot_objects.setdefault(slot_key, set()).add(
            record["object"].strip().lower()
        )

    grouped = {}

    for record in records:
        grouped.setdefault(
            record["triple_key"],
            record,
        )

    for triple_key, representative in grouped.items():
        slot_key = (
            representative["subject"].strip().lower(),
            representative["predicate"].strip().lower(),
        )

        object_value = representative["object"].strip().lower()
        competitors = len(
            slot_objects[slot_key] - {object_value}
        )

        confidences = collect_confidences(
            records,
            triple_key,
        )

        functional = is_functional(
            representative["predicate"]
        )

        support = conservative_noisy_or(
            confidences,
            shrinkage=DEFAULT_SHRINKAGE,
        )

        final_score = final_confidence(
            confidences=confidences,
            competitor_count=competitors,
            shrinkage=DEFAULT_SHRINKAGE,
            functional_predicate=functional,
        )

        rows.append({
            **representative,
            "observation_count": len(confidences),
            "observation_confidences": json.dumps(confidences),
            "functional_predicate": functional,
            "competitor_count": competitors,
            "support": support,
            "final_confidence": final_score,
            "threshold": 0.88,
            "above_threshold": final_score >= 0.88,
        })

    return pd.DataFrame(rows)

In [6]:
observation_rows = []

for triple in test_triples:
    observation_rows.append({
        "experiment_id": "DEC003_LOCAL_TEST",
        "run_id": "local_run_001",
        "iteration": 1,
        "subject": triple.subject,
        "predicate": triple.predicate,
        "object": triple.object,
        "triple_key": "|".join(triple.key()),
        "observation_confidence": triple.confidence,
        "source_id": triple.source_id,
        "source_type": triple.source_type,
        "provenance": triple.provenance,
        "extractor_version": triple.extractor_version,
    })

observations_df = pd.DataFrame(observation_rows)

scores_df = calculate_local_scores([
    {
        "experiment_id": "DEC003_LOCAL_TEST",
        "run_id": "local_run_001",
        "iteration": 1,
        "subject": triple.subject,
        "predicate": triple.predicate,
        "object": triple.object,
        "triple_key": "|".join(triple.key()),
        "confidence": triple.confidence,
        "source_id": triple.source_id,
        "source_type": triple.source_type,
        "provenance": triple.provenance,
    }
    for triple in test_triples
])

observations_path = OUTPUT_DIR / "observations_iteration_1.csv"
scores_path = OUTPUT_DIR / "pkb_snapshot_iteration_1.csv"

observations_df.to_csv(
    observations_path,
    index=False,
    encoding="utf-8",
)

scores_df.to_csv(
    scores_path,
    index=False,
    encoding="utf-8",
)

display(observations_df)
display(scores_df)

print("Saved observations:", observations_path)
print("Saved snapshot:", scores_path)
print("API calls: 0")

,experiment_id,run_id,iteration,subject,predicate,object,triple_key,observation_confidence,source_id,source_type,provenance,extractor_version
0,DEC003_LOCAL_TEST,local_run_001,1,Product-A,has_noise_cancellation,yes,product-a|has_noise_cancellation|yes,0.90,local-source-1,unstructured,Product-A supports noise cancellation.,local_test
1,DEC003_LOCAL_TEST,local_run_001,1,Product-A,has_noise_cancellation,yes,product-a|has_noise_cancellation|yes,0.92,local-source-2,unstructured,Noise cancellation is included for Product-A.,local_test
2,DEC003_LOCAL_TEST,local_run_001,1,Product-A,has_noise_cancellation,no,product-a|has_noise_cancellation|no,0.70,local-source-3,unstructured,Conflicting local test observation.,local_test
3,DEC003_LOCAL_TEST,local_run_001,1,Product-B,made_of_material,aluminum,product-b|made_of_material|aluminum,0.85,local-source-4,structured,Structured material field: aluminum.,local_test


,experiment_id,run_id,iteration,subject,predicate,object,triple_key,confidence,source_id,source_type,provenance,observation_count,observation_confidences,functional_predicate,competitor_count,support,final_confidence,threshold,above_threshold
0,DEC003_LOCAL_TEST,local_run_001,1,Product-A,has_noise_cancellation,yes,product-a|has_noise_cancellation|yes,0.90,local-source-1,unstructured,Product-A supports noise cancellation.,2,"[0.9, 0.92]",True,1,0.89925,0.449625,0.88,False
1,DEC003_LOCAL_TEST,local_run_001,1,Product-A,has_noise_cancellation,no,product-a|has_noise_cancellation|no,0.70,local-source-3,unstructured,Conflicting local test observation.,1,[0.7],True,1,0.52500,0.262500,0.88,False
2,DEC003_LOCAL_TEST,local_run_001,1,Product-B,made_of_material,aluminum,product-b|made_of_material|aluminum,0.85,local-source-4,structured,Structured material field: aluminum.,1,[0.85],False,0,0.63750,0.637500,0.88,False


Saved observations: E:\slde-aft\outputs\dec003_probkb_local_test\observations_iteration_1.csv
Saved snapshot: E:\slde-aft\outputs\dec003_probkb_local_test\pkb_snapshot_iteration_1.csv
API calls: 0


In [7]:
score_columns = [
    "subject",
    "predicate",
    "object",
    "observation_count",
    "observation_confidences",
    "functional_predicate",
    "competitor_count",
    "support",
    "final_confidence",
    "threshold",
    "above_threshold",
]

display(
    scores_df[score_columns]
    .sort_values(["subject", "predicate", "object"])
    .round({
        "support": 6,
        "final_confidence": 6,
    })
)

,subject,predicate,object,observation_count,observation_confidences,functional_predicate,competitor_count,support,final_confidence,threshold,above_threshold
1,Product-A,has_noise_cancellation,no,1,[0.7],True,1,0.52500,0.262500,0.88,False
0,Product-A,has_noise_cancellation,yes,2,"[0.9, 0.92]",True,1,0.89925,0.449625,0.88,False
2,Product-B,made_of_material,aluminum,1,[0.85],False,0,0.63750,0.637500,0.88,False


In [8]:
score_columns = [
    "subject",
    "predicate",
    "object",
    "observation_count",
    "observation_confidences",
    "functional_predicate",
    "competitor_count",
    "support",
    "final_confidence",
    "threshold",
    "above_threshold",
]

display(
    scores_df[score_columns]
    .sort_values(["subject", "predicate", "object"])
    .round({
        "support": 6,
        "final_confidence": 6,
    })
)

,subject,predicate,object,observation_count,observation_confidences,functional_predicate,competitor_count,support,final_confidence,threshold,above_threshold
1,Product-A,has_noise_cancellation,no,1,[0.7],True,1,0.52500,0.262500,0.88,False
0,Product-A,has_noise_cancellation,yes,2,"[0.9, 0.92]",True,1,0.89925,0.449625,0.88,False
2,Product-B,made_of_material,aluminum,1,[0.85],False,0,0.63750,0.637500,0.88,False


In [ ]:
yes_row = scores_df[
    (scores_df["subject"] == "Product-A")
    & (
        scores_df["predicate"]
        == "has_noise_cancellation"
    )
    & (scores_df["object"] == "yes")
].iloc[0]

no_row = scores_df[
    (scores_df["subject"] == "Product-A")
    & (
        scores_df["predicate"]
        == "has_noise_cancellation"
    )
    & (scores_df["object"] == "no")
].iloc[0]

material_row = scores_df[
    (scores_df["subject"] == "Product-B")
    & (scores_df["predicate"] == "made_of_material")
    & (scores_df["object"] == "aluminum")
].iloc[0]

expected_yes_support = 1.0 - (
    (1.0 - 0.75 * 0.90)
    * (1.0 - 0.75 * 0.92)
)

assert yes_row["observation_count"] == 2
assert bool(yes_row["functional_predicate"]) is True
assert yes_row["competitor_count"] == 1
assert yes_row["support"] == pytest.approx(
    expected_yes_support
)
assert yes_row["final_confidence"] == pytest.approx(
    expected_yes_support / 2.0
)

assert no_row["observation_count"] == 1
assert no_row["functional_predicate"] is True
assert no_row["competitor_count"] == 1
assert no_row["final_confidence"] == pytest.approx(
    no_row["support"] / 2.0
)

assert bool(material_row["functional_predicate"]) is False
assert material_row["competitor_count"] == 0
assert material_row["final_confidence"] == pytest.approx(
    material_row["support"]
)

assert scores_df["support"].between(0.0, 1.0).all()
assert scores_df["final_confidence"].between(0.0, 1.0).all()

print("PASS: Local DEC-003 score calculations are correct.")
print("PASS: Functional conflict penalty is correct.")
print("PASS: Non-functional predicate receives no conflict penalty.")
print("PASS: All scores are within [0, 1].")
print("API calls made: 0")

AssertionError: 